# 2D Toy Experiment: FQL vs FBRAC vs QFlow

This notebook trains and compares three flow-based RL agents on a 2D toy problem:
- **FQL**: Flow Q-Learning (simple baseline)
- **FBRAC**: Flow-based Behavioral Regularized Actor-Critic
- **QFlow**: Q-guided Flow with intermediate value function

The agents learn to match a target distribution (Swiss Roll or Two Spirals).

## Setup

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

from utils import generate_target_data, DOMAIN_MIN, DOMAIN_MAX, N_DATA, compute_mmd
from agents import train_fql, train_fbrac, train_qflow
from plotting import plot_comparison, plot_training_curves, plot_agent_samples

# Set up directories
SAVE_DIR = Path('results')
SAVE_DIR.mkdir(exist_ok=True)

print(f"JAX version: {jax.__version__}")
print(f"Device: {jax.devices()}")

## Generate Target Data

In [ ]:
# Choose dataset: 'swiss_roll' or 'two_spirals'
DATASET_TYPE = 'swiss_roll'

# Generate target data
key = jrandom.PRNGKey(0)
target_data = generate_target_data(DATASET_TYPE, n_samples=N_DATA, key=key)
target_data = np.array(target_data)

# Create reward signal based on target data
# Reward = -distance to nearest target point (approximately)
distances = np.linalg.norm(target_data[:, None, :] - target_data[None, :, :], axis=-1)
rewards = -np.mean(distances, axis=1)
rewards = (rewards - rewards.min()) / (rewards.max() - rewards.min())  # Normalize

print(f"Target data shape: {target_data.shape}")
print(f"Rewards shape: {rewards.shape}")
print(f"Reward range: [{rewards.min():.3f}, {rewards.max():.3f}]")

## Train Agents

In [ ]:
# Training configuration
EPOCHS = 2000
LR = 1e-3
FLOW_STEPS = 10
HIDDEN = 128
NUM_LAYERS = 4
BC_EPOCHS = 500
ALPHA_QFLOW = 0.5

print("Training FQL...")
fql_flow_params, fql_critic_params, fql_history = train_fql(
    target_data, rewards,
    epochs=EPOCHS,
    lr=LR,
    flow_steps=FLOW_STEPS,
    hidden=HIDDEN,
    num_layers=NUM_LAYERS,
)

print("\nTraining FBRAC...")
fbrac_flow_params, fbrac_critic_params, fbrac_history = train_fbrac(
    target_data, rewards,
    epochs=EPOCHS,
    lr=LR,
    flow_steps=FLOW_STEPS,
    hidden=HIDDEN,
    num_layers=NUM_LAYERS,
    bc_epochs=BC_EPOCHS,
)

print("\nTraining QFlow...")
qflow_flow_params, qflow_inner_critic_params, qflow_outer_critic_params, qflow_history = train_qflow(
    target_data, rewards,
    epochs=EPOCHS,
    lr=LR,
    flow_steps=FLOW_STEPS,
    hidden=HIDDEN,
    num_layers=NUM_LAYERS,
    bc_epochs=BC_EPOCHS,
    alpha=ALPHA_QFLOW,
)

print("\nTraining complete!")

## Generate Samples and Evaluate

In [ ]:
from utils import apply_mlp

# Helper function to integrate flows
def integrate_flow(flow_params, z, flow_forward_fn, steps=10):
    """Integrate flow from t=0 to t=1."""
    def step_fn(carry, _):
        x_curr, t_curr = carry
        dt = jnp.minimum(1.0 - t_curr, 1.0 / steps)
        v = flow_forward_fn(flow_params, x_curr, t_curr)
        x_next = x_curr + v * dt
        t_next = t_curr + dt
        x_next = jnp.clip(x_next, DOMAIN_MIN, DOMAIN_MAX)
        return (x_next, t_next), None
    (x_final, _), _ = jax.lax.scan(step_fn, (z, jnp.zeros((z.shape[0], 1))), None, length=steps)
    return x_final

def flow_forward(params, x, t):
    xt = jnp.concatenate([x, t], axis=-1)
    return apply_mlp(params, xt)

# Generate samples
N_SAMPLES = 512
key = jrandom.PRNGKey(999)
z = jrandom.normal(key, (N_SAMPLES, 2))

fql_samples = np.array(integrate_flow(fql_flow_params, z, flow_forward, steps=FLOW_STEPS))
fbrac_samples = np.array(integrate_flow(fbrac_flow_params, z, flow_forward, steps=FLOW_STEPS))
qflow_samples = np.array(integrate_flow(qflow_flow_params, z, flow_forward, steps=FLOW_STEPS))

print(f"Generated {N_SAMPLES} samples from each agent")

## Visualization

In [ ]:
# Plot comparison
fig = plot_comparison(target_data, fql_samples, fbrac_samples, qflow_samples)
plt.savefig(SAVE_DIR / f'{DATASET_TYPE}_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Plot training curves
fig = plot_training_curves(
    [fql_history, fbrac_history, qflow_history],
    figsize=(12, 4)
)
plt.savefig(SAVE_DIR / f'{DATASET_TYPE}_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## Quantitative Evaluation

In [ ]:
# Compute metrics
fql_mmd = compute_mmd(jnp.array(fql_samples), jnp.array(target_data))
fbrac_mmd = compute_mmd(jnp.array(fbrac_samples), jnp.array(target_data))
qflow_mmd = compute_mmd(jnp.array(qflow_samples), jnp.array(target_data))

print(f"\nMaximum Mean Discrepancy (MMD):")
print(f"FQL:   {fql_mmd:.4f}")
print(f"FBRAC: {fbrac_mmd:.4f}")
print(f"QFlow: {qflow_mmd:.4f}")

best_agent = ['FQL', 'FBRAC', 'QFlow'][np.argmin([fql_mmd, fbrac_mmd, qflow_mmd])]
print(f"\nBest agent: {best_agent}")

## Save Results

In [ ]:
# Save samples
np.save(SAVE_DIR / f'{DATASET_TYPE}_target.npy', target_data)
np.save(SAVE_DIR / f'{DATASET_TYPE}_fql_samples.npy', fql_samples)
np.save(SAVE_DIR / f'{DATASET_TYPE}_fbrac_samples.npy', fbrac_samples)
np.save(SAVE_DIR / f'{DATASET_TYPE}_qflow_samples.npy', qflow_samples)

# Save metrics
metrics = {
    'fql_mmd': float(fql_mmd),
    'fbrac_mmd': float(fbrac_mmd),
    'qflow_mmd': float(qflow_mmd),
}

import json
with open(SAVE_DIR / f'{DATASET_TYPE}_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Results saved to {SAVE_DIR}")